In [0]:

# 1. Configuration: Define the storage account and path for the Bronze layer (raw data)
storage_account = "stretailcdcproj"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/retail_orders/"

# 2. Data Ingestion: Read the transactional data from the Delta table
df_bronze = spark.read.format("delta").load(bronze_path)

# 3. Data Contract: Define the expected schema to enforce data quality for downstream use
expected_columns = [
    "order_id", "customer_id", "product", "region", 
    "quantity", "amount", "order_date", "last_modified"
]
actual_columns = df_bronze.columns

print("Expected columns:", expected_columns)
print("Actual columns:  ", actual_columns)

# 4. Schema Validation: Catch missing or unexpected structural changes early (schema drift)
if actual_columns == expected_columns:
    print("Schema check PASSED: columns match exactly")
else:
    # Isolate missing/extra columns to quickly identify the issue for debugging
    missing = set(expected_columns) - set(actual_columns)
    extra = set(actual_columns) - set(expected_columns)
    
    print("Schema check FAILED")
    print("Missing columns:", missing)
    print("Unexpected extra columns:", extra)

# 5. Data Type Verification: Display schema to ensure correct types (e.g., Integer vs String)
print("\nSchema details:")
df_bronze.printSchema()


Expected columns: ['order_id', 'customer_id', 'product', 'region', 'quantity', 'amount', 'order_date', 'last_modified']
Actual columns:   ['order_id', 'customer_id', 'product', 'region', 'quantity', 'amount', 'order_date', 'last_modified']
✅ Schema check PASSED: columns match exactly

Schema details:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- region: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- last_modified: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F

print("=" * 60)
print("DATA QUALITY VALIDATION REPORT")
print("=" * 60)

total_rows = df_bronze.count()
print(f"\nTotal rows in bronze: {total_rows}")

# ---- Check 1: Null Values ----
print("\n--- Null Value Check ---")
null_counts = df_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_bronze.columns
])
null_counts.show()

null_amount_count = df_bronze.filter(F.col("amount").isNull()).count()
print(f"⚠  Rows with NULL amount: {null_amount_count} ({null_amount_count/total_rows*100:.2f}%)")

# ---- Check 2: Duplicate Rows ----
print("\n--- Duplicate Check ---")
distinct_rows = df_bronze.distinct().count()
duplicate_count = total_rows - distinct_rows
print(f"⚠  Duplicate rows detected: {duplicate_count}")

# ---- Check 3: Duplicate order_id (Primary Key check) ----
print("\n--- Primary Key (order_id) Uniqueness Check ---")
distinct_order_ids = df_bronze.select("order_id").distinct().count()
duplicate_order_ids = total_rows - distinct_order_ids
print(f"⚠  Duplicate order_id count: {duplicate_order_ids}")

# ---- Check 4: Range Validation ----
print("\n--- Range Validation ---")
invalid_amount = df_bronze.filter((F.col("amount") < 0) | (F.col("amount") > 10000)).count()
invalid_quantity = df_bronze.filter((F.col("quantity") < 1) | (F.col("quantity") > 100)).count()
print(f"Rows with invalid amount (negative or >10000): {invalid_amount}")
print(f"Rows with invalid quantity (outside 1-100): {invalid_quantity}")

# ---- Summary ----
print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
total_issues = null_amount_count + duplicate_count + invalid_amount + invalid_quantity
print(f"Total rows scanned      : {total_rows}")
print(f"Rows with null amount   : {null_amount_count}")
print(f"Duplicate rows          : {duplicate_count}")
print(f"Invalid amount rows     : {invalid_amount}")
print(f"Invalid quantity rows   : {invalid_quantity}")
print(f"Total data quality flags: {total_issues}")

DATA QUALITY VALIDATION REPORT

Total rows in bronze: 5020

--- Null Value Check ---
+--------+-----------+-------+------+--------+------+----------+-------------+
|order_id|customer_id|product|region|quantity|amount|order_date|last_modified|
+--------+-----------+-------+------+--------+------+----------+-------------+
|       0|          0|      0|     0|       0|    56|         0|            0|
+--------+-----------+-------+------+--------+------+----------+-------------+

⚠  Rows with NULL amount: 56 (1.12%)

--- Duplicate Check ---
⚠  Duplicate rows detected: 20

--- Primary Key (order_id) Uniqueness Check ---
⚠  Duplicate order_id count: 20

--- Range Validation ---
⚠  Rows with invalid amount (negative or >10000): 0
⚠  Rows with invalid quantity (outside 1-100): 0

VALIDATION SUMMARY
Total rows scanned      : 5020
Rows with null amount   : 56
Duplicate rows          : 20
Invalid amount rows     : 0
Invalid quantity rows   : 0
Total data quality flags: 76
